# LLM Slot Extraction - Google Colab (Native llama.cpp + CUDA)

**Purpose**: Test Qwen3-8B-GGUF with native llama.cpp server on Colab GPU

**Requirements**:
- Google Colab with GPU runtime (T4 or better)
- ~6GB disk space for model

---

## Architecture

```
┌─────────────────┐    HTTP/JSON     ┌──────────────────┐
│  Python Client  │ ───────────────> │  llama-server    │
│  (this notebook)│                  │  (native, CUDA)  │
└─────────────────┘                  └──────────────────┘
                                              │
                                              ▼
                                     ┌──────────────────┐
                                     │ Qwen3-8B-GGUF    │
                                     │ (Q4_K_M, 5GB)    │
                                     └──────────────────┘
```

This is the same architecture as local macOS setup, but using CUDA instead of Metal.

## Step 0: Check GPU Availability

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Check CUDA version
!nvcc --version

## Step 1: Build llama.cpp from Source with CUDA

Since prebuilt CUDA binaries aren't available, we build from source (~3-5 minutes).

In [1]:
%%time
import os

# Clone llama.cpp
if not os.path.exists('/content/llama.cpp'):
    print("Cloning llama.cpp repository...")
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /content/llama.cpp
else:
    print("llama.cpp already cloned, pulling latest...")
    !cd /content/llama.cpp && git pull

print("\nRepository ready!")

Cloning llama.cpp repository...
fatal: could not create leading directories of '/content/llama.cpp': Read-only file system

Repository ready!
CPU times: user 12.8 ms, sys: 8.84 ms, total: 21.6 ms
Wall time: 309 ms


In [ ]:
%%time
# Build with CUDA support
print("Building llama.cpp with CUDA support...")
print("This takes 3-5 minutes on Colab.\n")

!cd /content/llama.cpp && \
    cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES="75;80;86" && \
    cmake --build build --config Release -j$(nproc)

print("\nBuild complete!")

In [ ]:
# Verify build
import os
import glob

# Find llama-server
search_paths = [
    "/content/llama.cpp/build/bin/llama-server",
    "/content/llama.cpp/build/llama-server",
    "/content/llama.cpp/llama-server",
]

LLAMA_SERVER_PATH = None
for path in search_paths:
    if os.path.exists(path):
        LLAMA_SERVER_PATH = path
        break

# Also search recursively
if not LLAMA_SERVER_PATH:
    matches = glob.glob("/content/llama.cpp/**/llama-server", recursive=True)
    if matches:
        LLAMA_SERVER_PATH = matches[0]

if LLAMA_SERVER_PATH:
    LLAMA_LIB_PATH = os.path.dirname(LLAMA_SERVER_PATH)
    print(f"Found llama-server at: {LLAMA_SERVER_PATH}")
    print(f"Library path: {LLAMA_LIB_PATH}")
    
    # Make executable
    os.chmod(LLAMA_SERVER_PATH, 0o755)
    
    # Test version
    print("\nVersion info:")
    !{LLAMA_SERVER_PATH} --version 2>&1 || echo "(version check done)"
else:
    print("ERROR: llama-server not found!")
    print("\nSearching for executables:")
    !find /content/llama.cpp/build -type f -executable -name "llama*" 2>/dev/null

## Step 2: Download Qwen3-8B-GGUF Model

In [ ]:
# Install huggingface_hub for downloading
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import hf_hub_download
import os
import glob

MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_FILE = "Qwen3-8B-Q4_K_M.gguf"
MODEL_PATH = f"{MODEL_DIR}/{MODEL_FILE}"

# Check if model exists (case-insensitive)
existing_models = glob.glob(f"{MODEL_DIR}/*q4_k_m*.gguf", recursive=False) + \
                  glob.glob(f"{MODEL_DIR}/*Q4_K_M*.gguf", recursive=False)

if existing_models:
    MODEL_PATH = existing_models[0]
    print(f"Model already exists: {MODEL_PATH}")
    print(f"Size: {os.path.getsize(MODEL_PATH) / 1e9:.2f} GB")
else:
    print(f"Downloading Qwen3-8B Q4_K_M from HuggingFace...")
    print("This may take 5-15 minutes depending on connection speed.\n")
    
    model_path = hf_hub_download(
        repo_id="Qwen/Qwen3-8B-GGUF",
        filename=MODEL_FILE,
        local_dir=MODEL_DIR
    )
    MODEL_PATH = model_path
    
    print(f"\nDownload complete: {MODEL_PATH}")
    print(f"Size: {os.path.getsize(MODEL_PATH) / 1e9:.2f} GB")

## Step 3: Start llama-server with CUDA

In [ ]:
import subprocess
import time
import requests
import os

# Server configuration
SERVER_PORT = 8080
CONTEXT_SIZE = 4096
GPU_LAYERS = 99  # Offload all layers to GPU

# Verify paths
if not LLAMA_SERVER_PATH or not os.path.exists(LLAMA_SERVER_PATH):
    raise FileNotFoundError(f"llama-server not found. Please re-run Step 1.")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found at {MODEL_PATH}. Please re-run Step 2.")

print("Starting llama-server with CUDA acceleration...")
print(f"Binary: {LLAMA_SERVER_PATH}")
print(f"Model: {MODEL_PATH}")
print(f"Context: {CONTEXT_SIZE}")
print(f"GPU Layers: {GPU_LAYERS}")
print()

# Kill any existing server
!pkill -f llama-server 2>/dev/null || true
time.sleep(2)

# Set library path
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = f"{LLAMA_LIB_PATH}:/usr/local/cuda/lib64:" + env.get("LD_LIBRARY_PATH", "")

# Start server
server_process = subprocess.Popen(
    [
        LLAMA_SERVER_PATH,
        "--model", MODEL_PATH,
        "--ctx-size", str(CONTEXT_SIZE),
        "--n-gpu-layers", str(GPU_LAYERS),
        "--port", str(SERVER_PORT),
        "--host", "0.0.0.0"
    ],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait for server to start
print("Waiting for server to start (loading model into GPU)...")
server_ready = False
for i in range(120):  # Wait up to 2 minutes
    time.sleep(1)
    try:
        response = requests.get(f"http://localhost:{SERVER_PORT}/health", timeout=2)
        if response.status_code == 200:
            print(f"\n\nServer is ready at http://localhost:{SERVER_PORT}")
            print(f"Health: {response.json()}")
            server_ready = True
            break
    except:
        if i % 10 == 0:
            print(f"  {i}s...", end="", flush=True)
        else:
            print(".", end="", flush=True)

if not server_ready:
    print("\n\nERROR: Server failed to start!")
    print("\nServer output (last 3000 chars):")
    server_process.terminate()
    try:
        output = server_process.stdout.read()
        print(output[-3000:] if len(output) > 3000 else output)
    except:
        print("Could not read output")

In [ ]:
# Check server health and GPU memory
import requests

SERVER_URL = f"http://localhost:{SERVER_PORT}"

try:
    response = requests.get(f"{SERVER_URL}/health", timeout=5)
    print(f"Server Status: {response.status_code}")
    print(f"Health: {response.json()}")
except Exception as e:
    print(f"Error: {e}")

print("\nGPU Memory Usage:")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## Step 4: Define Slot Extraction Client

In [ ]:
import json
import re
import requests
from typing import Dict, Any, Optional

# System prompt for slot extraction
SYSTEM_PROMPT = """You are an e-commerce chatbot assistant for Bagisto. Your task is to extract structured information from customer queries in Bahasa Indonesia.

Output ONLY valid JSON with these fields:
{
  "task": "check_order" | "ask_price" | "check_stock" | "ask_payment" | "product_info" | "out_of_scope",
  "entities": {
    "product_name": "string or null",
    "order_id": "string or null",
    "payment_method": "string or null",
    "variant": "string or null",
    "quantity": "number or null"
  },
  "multi_intent": ["list of secondary intents if any"],
  "confidence": 0.0-1.0,
  "needs_clarification": true | false,
  "clarification_question": "string or null if needs_clarification is true"
}

Task definitions:
- check_order: Customer asking about order status, tracking, delivery time
- ask_price: Customer asking about product price, discount, promo
- check_stock: Customer asking if product is available, in stock
- ask_payment: Customer asking about payment methods, how to pay
- product_info: Customer asking about product details, specs, description
- out_of_scope: Refund, complaint, store info, or unrelated questions

Handle informal Indonesian:
- lo, gue, gw → customer self-reference
- gak, ga, nggak → tidak (no/not)
- gimana, gmn → bagaimana (how)
- dong, sih, deh → emphasis particles
- brp, hrg → berapa, harga
- psen, psn → pesanan

IMPORTANT: Output ONLY the JSON object, no explanation or additional text."""


class SlotExtractor:
    """Client for llama-server slot extraction."""
    
    def __init__(self, server_url: str = "http://localhost:8080"):
        self.server_url = server_url
        self.api_url = f"{server_url}/v1/chat/completions"
    
    def is_server_running(self) -> bool:
        """Check if llama-server is running."""
        try:
            response = requests.get(f"{self.server_url}/health", timeout=5)
            return response.status_code == 200
        except:
            return False
    
    def extract(self, query: str, temperature: float = 0.1, verbose: bool = False) -> Dict[str, Any]:
        """Extract slots from Indonesian e-commerce query."""
        payload = {
            "model": "qwen3-8b",
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": query}
            ],
            "max_tokens": 512,
            "temperature": temperature,
            "top_p": 0.9,
            "response_format": {"type": "json_object"}
        }
        
        try:
            response = requests.post(self.api_url, json=payload, timeout=60)
            response.raise_for_status()
            
            data = response.json()
            raw_content = data['choices'][0]['message']['content']
            
            if verbose:
                print(f"Raw response: {raw_content}")
            
            # Parse JSON from response
            json_match = re.search(r'\{[\s\S]*\}', raw_content)
            if json_match:
                result = json.loads(json_match.group())
            else:
                result = json.loads(raw_content)
            
            return result
            
        except requests.exceptions.ConnectionError:
            return {"task": "out_of_scope", "entities": {}, "error": "Server not running"}
        except json.JSONDecodeError as e:
            return {"task": "out_of_scope", "entities": {}, "error": f"JSON parse error: {str(e)}"}
        except Exception as e:
            return {"task": "out_of_scope", "entities": {}, "error": str(e)}


# Initialize extractor
extractor = SlotExtractor(server_url=SERVER_URL)
print(f"SlotExtractor initialized!")
print(f"Server URL: {extractor.server_url}")
print(f"Server running: {extractor.is_server_running()}")

## Step 5: Test Slot Extraction

In [ ]:
# Test queries
test_queries = [
    # Stock
    "stok jaket biru masih ada ga?",
    "ready stock sepatu nike size 42?",
    
    # Price
    "harga tas ransel berapa?",
    "ada diskon gak untuk laptop asus?",
    
    # Order status
    "pesanan saya 12345 udah sampai mana?",
    "order gue kapan nyampe ya",
    
    # Payment
    "bisa bayar pake gopay ga?",
    "cara bayar gimana sih",
    
    # Product info
    "spek laptop lenovo yang ini gimana?",
    "bahan kaos ini apa ya",
    
    # Multi-intent
    "stok hp samsung ada ga? kalo ada harga berapa?",
    "jaket ini ready ga? gimana cara bayarnya?",
    
    # Out of scope
    "mau refund dong barang rusak",
    "jam buka toko kapan ya",
    
    # Informal/typo
    "brp hrg sepatu nike?",
    "psen gw 99871 gmn?",
]

print("Testing slot extraction with native llama.cpp (CUDA)...")
print("=" * 60)

import time
results = []
latencies = []

for query in test_queries:
    print(f"\nQuery: \"{query}\"")
    
    start = time.time()
    result = extractor.extract(query)
    elapsed = time.time() - start
    latencies.append(elapsed)
    
    results.append({"query": query, "result": result, "latency": elapsed})
    
    if "error" in result:
        print(f"  ERROR: {result['error']}")
        break
    else:
        print(f"  Task: {result.get('task', 'N/A')}")
        print(f"  Entities: {result.get('entities', {})}")
        if result.get('multi_intent'):
            print(f"  Multi-intent: {result.get('multi_intent')}")
        print(f"  Confidence: {result.get('confidence', 'N/A')}")
        print(f"  Latency: {elapsed:.2f}s")

print("\n" + "=" * 60)
print(f"Tested {len(results)} queries")

## Step 6: Latency Benchmark

In [ ]:
import numpy as np

if latencies:
    print("Latency Statistics (Native llama.cpp + CUDA):")
    print(f"  Mean: {np.mean(latencies):.3f}s")
    print(f"  Std: {np.std(latencies):.3f}s")
    print(f"  Min: {np.min(latencies):.3f}s")
    print(f"  Max: {np.max(latencies):.3f}s")
    print(f"  Median: {np.median(latencies):.3f}s")
    print(f"\nThroughput: ~{60/np.mean(latencies):.1f} queries/minute")
else:
    print("No latency data - check if server is running")

## Step 7: Interactive Testing

In [ ]:
# Change this query to test different inputs
query = "stok sepatu nike air jordan size 42 ada ga? kalo ada berapa harganya?"

print(f"Query: \"{query}\"")
print("-" * 50)

start = time.time()
result = extractor.extract(query, verbose=True)
elapsed = time.time() - start

print(f"\nLatency: {elapsed:.2f}s")
print("\nParsed Result:")
print(json.dumps(result, indent=2, ensure_ascii=False))

## Step 8: Save Results

In [ ]:
from datetime import datetime

if results and "error" not in results[0].get("result", {}):
    # Get GPU info
    gpu_name = "Unknown"
    try:
        import torch
        if torch.cuda.is_available():
            gpu_name = torch.cuda.get_device_name(0)
    except:
        pass
    
    test_summary = {
        "timestamp": datetime.now().isoformat(),
        "model": "Qwen3-8B-GGUF (Q4_K_M)",
        "runtime": f"Google Colab - {gpu_name}",
        "backend": "Native llama.cpp + CUDA (built from source)",
        "total_queries": len(results),
        "latency_stats": {
            "mean": float(np.mean(latencies)),
            "std": float(np.std(latencies)),
            "min": float(np.min(latencies)),
            "max": float(np.max(latencies)),
            "median": float(np.median(latencies))
        },
        "results": results
    }

    output_file = "llm_slot_extraction_colab_results.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(test_summary, f, indent=2, ensure_ascii=False)

    print(f"Results saved to: {output_file}")
    
    # Download
    from google.colab import files
    files.download(output_file)
else:
    print("No results to save - check if server is running")

## Step 9: Cleanup

In [ ]:
# Stop the server when done
if 'server_process' in dir() and server_process:
    server_process.terminate()
    print("Server stopped.")

## Summary

### Setup (Build from Source on Colab)

```bash
# 1. Clone llama.cpp
git clone --depth 1 https://github.com/ggerganov/llama.cpp.git

# 2. Build with CUDA
cmake -B build -DGGML_CUDA=ON
cmake --build build --config Release -j$(nproc)

# 3. Download model & start server
# 4. Use HTTP API (same as local)
```

### Configuration
- **Model**: Qwen3-8B-GGUF (Q4_K_M, 4-bit)
- **Size**: ~5 GB
- **Backend**: Native llama.cpp (built from source with CUDA)
- **Acceleration**: CUDA (Colab GPU)
- **Context**: 4096 tokens
- **API**: OpenAI-compatible at http://localhost:8080

### Expected Performance on Colab T4
- **Build time**: ~3-5 minutes
- **Latency**: 0.3-1.0s per query
- **Throughput**: ~60-200 queries/minute
- **Memory**: ~5-6 GB VRAM

### Comparison with Local (macOS M1 Pro)
| Aspect | Colab (T4 GPU) | Local (M1 Pro) |
|--------|----------------|----------------|
| Backend | Native llama.cpp + CUDA | Native llama.cpp + Metal |
| Setup | Build from source (~5min) | brew install llama.cpp |
| Latency | ~0.3-1.0s | ~0.3-1.5s |
| Persistence | Session-based | Permanent |